<a href="https://colab.research.google.com/github/nancymatijas/alumni-llm-workshop/blob/main/01_llm_api_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM API Fundamentals: Build an AI Quiz Game

## Overview

In this notebook, we'll learn the fundamentals of interacting with Large Language Models through APIs by building an AI-powered quiz game, step by step.

We'll explore:
- Making API calls and inspecting responses
- Understanding that models are **stateless** — the full conversation must be sent every time
- How system prompts control model behavior
- Getting **structured output** (XML) and parsing it in Python
- **Thinking models** that show their reasoning (and burn tokens doing it)
- Building a complete **Study Buddy** quiz game from your own notes

**What You'll Build:** An AI-powered quiz game that generates questions from YOUR study materials.

**API:** We use [OpenRouter](https://openrouter.ai/), which gives access to many LLM providers through a single OpenAI-compatible API.

**Time:** ~2 hours (6 exercises)

## Setup and Installation

First, let's install the required packages and set up our environment.

In [34]:
# Install required packages
!pip install openai -q

In [35]:
import os
import time
import xml.etree.ElementTree as ET
from openai import OpenAI
from google.colab import userdata

# Configure OpenRouter API
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', None)

if not OPENROUTER_API_KEY:
    raise ValueError("Please set your OPENROUTER_API_KEY in Colab secrets (key icon in sidebar).")

# Initialize OpenAI client with OpenRouter endpoint
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Models we'll use throughout the notebook
MODEL_FAST = "stepfun/step-3.5-flash:free"                  # Fast and free
MODEL_MISTRAL = "mistralai/mistral-small-2603"               # Balanced
MODEL_NEMOTRON = "nvidia/nemotron-3-super-120b-a12b:free"    # Large open-weight model
MODEL_THINKING = "openai/gpt-oss-120b"                       # Reasoning/thinking model
MODEL_MINISTRAL = "mistralai/ministral-14b-2512"             # Mid-size Mistral

print("\u2713 OpenRouter client configured")
print(f"\u2713 Models available:")
print(f"  \u2022 {MODEL_FAST} (fast, free)")
print(f"  \u2022 {MODEL_MISTRAL} (balanced)")
print(f"  \u2022 {MODEL_NEMOTRON} (large open-weight)")
print(f"  \u2022 {MODEL_THINKING} (reasoning/thinking)")
print(f"  \u2022 {MODEL_MINISTRAL} (mid-size Mistral)")

✓ OpenRouter client configured
✓ Models available:
  • stepfun/step-3.5-flash:free (fast, free)
  • mistralai/mistral-small-2603 (balanced)
  • nvidia/nemotron-3-super-120b-a12b:free (large open-weight)
  • openai/gpt-oss-120b (reasoning/thinking)
  • mistralai/ministral-14b-2512 (mid-size Mistral)


In [36]:
def chat(messages, model=MODEL_FAST, temperature=0.7, **kwargs):
    """
    Send messages to an LLM and return the full response object.

    Args:
        messages: List of message dicts with 'role' and 'content'
        model: Model identifier string
        temperature: Sampling temperature (0.0 = deterministic, 1.0+ = creative)
        **kwargs: Additional parameters passed to the API

    Returns:
        The full API response object
    """
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        **kwargs
    )
    return response

print("✓ chat() helper function defined")

✓ chat() helper function defined


---

## Exercise 1: Hello, API

### Goal
Make your first LLM API call, understand the request/response structure, and compare how different models respond to the same prompt.

### Key Concepts
- The Chat Completions API uses a `messages` array
- Each message has a `role` (`"user"`, `"assistant"`, or `"system"`) and `content`
- The response contains the generated text, model info, and **token usage**

In [37]:
# Your first API call!
messages = [
    {"role": "user", "content": "What is an API? Explain in 2 sentences."}
]

response = chat(messages)

# Let's inspect the FULL response object — this isn't magic, it's a data structure
print("=== Full Response Object ===")
print(f"ID:      {response.id}")
print(f"Model:   {response.model}")
print(f"Created: {response.created}")
print()

# The actual generated text
print("=== Generated Text ===")
print(response.choices[0].message.content)
print()

# Token usage — this is how you get billed!
print("=== Token Usage ===")
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens:      {response.usage.total_tokens}")
print()

import json
print("=== Full JSON Response ===")
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

=== Full Response Object ===
ID:      gen-1774951722-w0XjqSAtMwGmIketclH8
Model:   stepfun/step-3.5-flash:free
Created: 1774951722

=== Generated Text ===
An API (Application Programming Interface) is a set of rules and tools that allows different software applications to communicate and exchange data with each other. It acts as an intermediary, defining how requests and responses should be made, enabling developers to use specific functionalities—like accessing a web service or operating system features—without needing to understand their internal code.

=== Token Usage ===
Prompt tokens:     23
Completion tokens: 175
Total tokens:      198

=== Full JSON Response ===
{
  "id": "gen-1774951722-w0XjqSAtMwGmIketclH8",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "An API (Application Programming Interface) is a set of rules and tools that allows different software applications to communicate and exchan

### Understanding Tokens

Tokens are the "units" that LLMs process. They're not exactly words — roughly:
- **English:** 100 tokens ≈ 75 words
- **Croatian:** 100 tokens ≈ 40 words (non-English languages use more tokens!)
- `"Hello world"` = 2 tokens
- `"Antiestablishmentarianism"` = 6 tokens

**Why tokens matter:**
1. You **pay per token** (input + output)
2. Every model has a maximum **context window** (total tokens for input + output)
3. Different models have different tokenizers → different token counts for the same text

In [38]:
# Same prompt, different models — compare responses, tokens, and speed
prompt = "Explain recursion to a 10-year-old in 3 sentences."

models = [MODEL_FAST, MODEL_MISTRAL, MODEL_NEMOTRON, MODEL_MINISTRAL]

for model_name in models:
    print(f"\n{'=' * 70}")
    print(f"Model: {model_name}")
    print('=' * 70)

    start_time = time.time()
    response = chat(
        [{"role": "user", "content": prompt}],
        model=model_name
    )
    elapsed = time.time() - start_time

    print(f"\nResponse:")
    print(response.choices[0].message.content)
    print(f"\nTokens: {response.usage.prompt_tokens} in + {response.usage.completion_tokens} out = {response.usage.total_tokens} total")
    print(f"Time: {elapsed:.2f}s")


Model: stepfun/step-3.5-flash:free

Response:
Imagine a set of Russian nesting dolls—each doll opens to reveal a smaller doll inside, and that one opens to an even smaller one, until you find the tiniest doll that can't open anymore. Recursion is like that: a task or question that keeps calling a smaller version of itself over and over. It helps solve big problems by breaking them into tiny, repeatable steps until you reach the simplest answer.

Tokens: 25 in + 287 out = 312 total
Time: 8.36s

Model: mistralai/mistral-small-2603

Response:
Recursion is when a problem is solved by breaking it down into smaller, similar problems—like a Russian nesting doll where each doll is a tiny version of the bigger one. For example, to count down from 5, you’d count down from 4 first, then 3, and so on until you reach 1. It’s like giving instructions to yourself step by step until you hit the simplest case!

Tokens: 30 in + 85 out = 115 total
Time: 0.87s

Model: nvidia/nemotron-3-super-120b-a12b:fr

### Observations

Notice how:
- Different models give **different answers** to the same prompt
- Token counts vary (different tokenizers!)
- Response time varies (model size, provider load)
- All use the **exact same API interface** — that's the power of OpenRouter

### Try It Yourself
Change the prompt in the cell below and re-run!

In [39]:
# YOUR TURN: Try your own prompt!
my_prompt = "Napiši mi haiku o siru. Na hrvatskom."  # <-- Change this!

response = chat([{"role": "user", "content": my_prompt}], model=MODEL_NEMOTRON)
print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

Sirje topli san,  
miris livade у устах,  
srca slatko bije.

Tokens used: 615


---

## Exercise 2: The Blank Slate

### Goal
Understand that LLMs have **NO memory** between API calls. Each call is completely independent. To have a "conversation," **YOU** must send the full history every time.

### Key Insight
The model is like talking to someone with amnesia — every API call is a fresh start. There is no session, no cookies, no state on the server. The `messages` array **IS** the memory.

In [40]:
# Call 1: Tell the model your name
response1 = chat([
    {"role": "user", "content": "My name is Alice and I study Computer Science at FERIT."}
])
print("Call 1 — Model's response:")
print(response1.choices[0].message.content)

print(f"\n{'=' * 70}")

# Call 2: Ask it what your name is (SEPARATE API call!)
response2 = chat([
    {"role": "user", "content": "What is my name and what do I study?"}
])
print("\nCall 2 — Model's response:")
print(response2.choices[0].message.content)
print("\n→ The model has NO IDEA! Each call is independent. The slate is blank.")

Call 1 — Model's response:
Hello, Alice! 👋 It's great to meet you. FERIT (Faculty of Electrical Engineering, Computer Science and Information Technology Osijek) is a well-known institution in Croatia for tech studies. 

How can I assist you today? Whether you have questions about programming, algorithms, projects, coursework, or just want to chat about Computer Science, I'm here to help! 😊


Call 2 — Model's response:
I don’t have access to personal information about you unless you share it with me during our conversation. Since you haven’t told me your name or what you study, I don’t know those details.

If you’d like, you can tell me your name and what you’re studying, and I’ll remember it for the rest of our current chat! 😊

→ The model has NO IDEA! Each call is independent. The slate is blank.


### The Solution: Send the Full Conversation History

To maintain context, you must include the **entire conversation** in each request:

```python
messages = [
    {"role": "user", "content": "My name is Alice..."},        # Turn 1
    {"role": "assistant", "content": "Nice to meet you..."},   # Model's reply to turn 1
    {"role": "user", "content": "What is my name?"},           # Turn 2
]
```

The model sees ALL messages and can "remember" the conversation. But it's not real memory — it's just reading the text you sent.

In [41]:
# Now let's do it RIGHT — include the full conversation history
messages_with_history = [
    {"role": "user", "content": "My name is Alice and I study Computer Science at FERIT."},
    {"role": "assistant", "content": response1.choices[0].message.content},  # include previous response!
    {"role": "user", "content": "What is my name and what do I study?"}
]

response3 = chat(messages_with_history)
print("With full history, the model responds:")
print(response3.choices[0].message.content)
print("\n→ Now it knows! Because we sent the ENTIRE conversation.")
print(f"\nWe sent {len(messages_with_history)} messages ({response3.usage.prompt_tokens} prompt tokens)")

With full history, the model responds:
Your name is **Alice**, and you study **Computer Science at FERIT** (Faculty of Electrical Engineering, Computer Science and Information Technology Osijek). 😊

→ Now it knows! Because we sent the ENTIRE conversation.

We sent 3 messages (127 prompt tokens)


In [42]:
class Conversation:
    """
    Manages a multi-turn conversation with an LLM.
    Accumulates messages so the model has full context on every call.
    """

    def __init__(self, model=MODEL_FAST, system_prompt=None):
        self.model = model
        self.messages = []
        self.total_tokens = 0
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})

    def say(self, user_message, **kwargs):
        """Send a message and get a response. History is maintained automatically."""
        self.messages.append({"role": "user", "content": user_message})

        response = chat(self.messages, model=self.model, **kwargs)
        assistant_message = response.choices[0].message.content

        self.messages.append({"role": "assistant", "content": assistant_message})

        self.total_tokens += response.usage.total_tokens

        return assistant_message, self.total_tokens

    def get_history(self):
        """Return the full message history."""
        return self.messages.copy()

print("✓ Conversation class defined")

✓ Conversation class defined


In [60]:
# Multi-turn conversation using our helper
conv = Conversation()

print("Turn 1:")
reply, usage = conv.say("My name is Bob. I'm learning about LLMs.")
print(reply)
print(f"  [{conv.total_tokens} tokens]")

print(f"\n{'=' * 70}\nTurn 2:")
reply, usage = conv.say("What am I learning about?")
print(reply)
print(f"  [{conv.total_tokens} tokens]")

print(f"\n{'=' * 70}\nTurn 3:")
reply, usage = conv.say("Summarize our conversation so far in one sentence.")
print(reply)
print(f"  [{conv.total_tokens} tokens]")

print(f"\n{'=' * 70}")
print(f"Messages in history: {len(conv.get_history())}")
print("→ Notice how token count GROWS with each turn — we're re-sending everything!")

Turn 1:
Hi Bob! 👋 Welcome to the world of LLMs — that's **Large Language Models**. 

Think of them as supercharged autocomplete systems trained on massive amounts of text from books, websites, and other sources. They learn patterns in language so well that they can:

- Answer questions
- Write essays or code
- Summarize text
- Translate languages
- Hold a conversation (like we're doing now!)

Some popular examples you might have heard of are **GPT-4** (what I’m based on), **Claude**, **Gemini**, and open-source models like **Llama**.

---

**A few fun facts to start:**
1. They don’t “understand” like humans — they predict the next most likely word based on patterns.
2. They can sometimes make up facts (called *hallucinations*), so it’s good to double-check important info.
3. They’re used in chatbots, coding assistants, content creation tools, and much more.

---

What part of LLMs are you most curious about?  
For example:
- How they’re trained?
- How to use them effectively?
- Their e

In [44]:
# What happens when we remove a message from the middle?
print("=== Full history (working) ===")
full_history = [
    {"role": "user", "content": "The secret code is BLUE42. Remember it."},
    {"role": "assistant", "content": "Got it! I've noted the secret code BLUE42."},
    {"role": "user", "content": "Now tell me: what is a binary tree?"},
    {"role": "assistant", "content": "A binary tree is a data structure where each node has at most two children, called left and right."},
    {"role": "user", "content": "What was the secret code I told you earlier?"}
]
resp_full = chat(full_history)
print(resp_full.choices[0].message.content)

print(f"\n{'=' * 70}")
print("=== History with first exchange removed (broken) ===")
broken_history = [
    {"role": "user", "content": "Now tell me: what is a binary tree?"},
    {"role": "assistant", "content": "A binary tree is a data structure where each node has at most two children, called left and right."},
    {"role": "user", "content": "What was the secret code I told you earlier?"}
]
resp_broken = chat(broken_history)
print(resp_broken.choices[0].message.content)
print("\n→ Without the earlier messages, the model has no idea about the secret code!")

=== Full history (working) ===
The secret code is **BLUE42**.

=== History with first exchange removed (broken) ===
I don't have any memory of previous conversations or shared information between sessions. Each chat is isolated, and I don't store personal data or secrets from earlier interactions.

If you're referring to something you mentioned **in this current conversation**, you haven't shared any secret code with me yet. If you'd like to share one now for context, feel free to do so, and I'll do my best to help based on what you provide! 🔐

*(Just a friendly reminder: avoid sharing actual sensitive passwords, keys, or personal secrets in any chat.)*

→ Without the earlier messages, the model has no idea about the secret code!


---

## Exercise 3: System Prompts & Personas

### Goal
Learn how the **system prompt** controls the model's behavior, personality, and response style.

### Key Concept
The `system` role sets instructions **before** the conversation begins. The model treats it as its identity and operating instructions.

```python
messages = [
    {"role": "system", "content": "You are a helpful tutor..."},   # ← THIS controls behavior
    {"role": "user", "content": "Explain APIs to me"}
]
```

In [45]:
# Same question, 4 different system prompts — watch the output change dramatically
user_question = "What is an API?"

system_prompts = {
    "No system prompt": None,
    "Patient Tutor": "You are a patient university tutor. Explain concepts simply using everyday analogies. Keep answers under 3 sentences.",
    "Software Architect": "You are a senior software architect with 20 years of experience. Give precise, technical answers. Use proper terminology. Be concise.",
    "Pirate Captain": "You are a pirate captain who explains technology using nautical metaphors. Stay in character at all times! Arrr!"
}

for name, sys_prompt in system_prompts.items():
    messages = []
    if sys_prompt:
        messages.append({"role": "system", "content": sys_prompt})
    messages.append({"role": "user", "content": user_question})

    response = chat(messages)

    print(f"\n{'=' * 70}")
    print(f"PERSONA: {name}")
    print('=' * 70)
    if sys_prompt:
        print(f"System: \"{sys_prompt[:80]}...\"")
    print(f"\n{response.choices[0].message.content}")


PERSONA: No system prompt

Excellent question! An **API (Application Programming Interface)** is one of the most fundamental and powerful concepts in modern software development.

In the simplest terms:

**An API is a set of rules, protocols, and tools that allows different software applications to communicate and share data with each other.**

Think of it as a **contract** or a **waiter in a restaurant**:

1.  **You (the Client):** You are one application (e.g., your mobile weather app).
2.  **The Kitchen (the Server):** This is another application or service that has the data or functionality you need (e.g., a weather data provider like AccuWeather).
3.  **The Waiter (the API):** You don't go into the kitchen yourself. You give your order (a **request**) to the waiter (the API). The waiter knows the kitchen's procedures, takes your order, fetches the food (the **data/response**), and brings it back to you in a standardized format you can understand (your meal on a plate).

---

### 

### Your Turn: Craft a Quiz Master System Prompt

Now create a system prompt for a **Quiz Master** — we'll use this later!

A good quiz master prompt should specify:
- The role (you are a quiz master)
- The tone (encouraging? strict?)
- Constraints (difficulty level, question style)
- We'll add the output format in Exercise 4

In [46]:
# YOUR TURN: Write a quiz master system prompt
QUIZ_MASTER_PROMPT = """You are a Quiz Master for university students.
Your job is to generate challenging but fair quiz questions.
- Questions should test understanding, not just memorization
- Always provide 4 options (A, B, C, D)
- Include a brief explanation for the correct answer
- Be encouraging and educational in tone
"""

# Test it!
messages = [
    {"role": "system", "content": QUIZ_MASTER_PROMPT},
    {"role": "user", "content": "Generate a quiz question about Python programming."}
]
response = chat(messages)
print(response.choices[0].message.content)
print("\n→ It works! But how do we PARSE this in code? That's Exercise 4...")

**Question:**  
What is the output of the following Python code?

```python
def func(x=[]):
    x.append(1)
    return x

print(func())
print(func())
print(func([]))
```

**Options:**  
A) `[1]`, `[1, 1]`, `[1]`  
B) `[1]`, `[1, 1]`, `[1, 1]`  
C) `[1]`, `[1]`, `[1]`  
D) `[]`, `[1]`, `[1]`

---

**Answer & Explanation:**  
✅ **A) `[1]`, `[1, 1]`, `[1]`**

**Why?**  
- In Python, **default argument values are evaluated only once**—when the function is defined, not each time it’s called.  
- `x=[]` creates a single list object at function definition. Subsequent calls to `func()` without an argument reuse that same list, so `append()` adds to the existing list.  
- The third call `func([])` explicitly passes a new empty list, so it starts fresh and appends `1` to that new list.

💡 **Key takeaway:** Avoid mutable default arguments (like `[]` or `{}`) unless you intentionally want shared state. Use `None` and initialize inside the function instead:

```python
def func(x=None):
    if x is 

---

## Exercise 4: Structured Output & Parsing

### Goal
Learn to get LLMs to produce **machine-parseable output**. This is essential for building real applications — you need to **extract data** from LLM responses, not just display text.

### The Problem
In Exercise 3, the quiz master generated a nice-looking quiz question. But it's just free-form text. Our quiz game needs **structured data**: the question, the options, the correct answer, the explanation — all as **separate fields** we can work with in code.

### Our Approach: XML
We'll use XML tags because:
- LLMs are very good at generating well-formed XML
- Python has built-in XML parsing (`xml.etree.ElementTree`)
- Clear opening/closing tags reduce parsing errors
- It's more robust than trying to regex free-form text

In [47]:
# The WRONG way: ask for a quiz question without specifying format
response = chat([
    {"role": "user", "content": "Generate a quiz question about computer networks with 4 options and the correct answer."}
])

raw_text = response.choices[0].message.content
print("=== Raw LLM Output ===")
print(raw_text)
print("\n" + "=" * 70)
print("THE PROBLEM: How do we extract the question, options, and correct")
print("answer from this? Every model formats it differently. Every run might")
print("be different. Parsing free-form text is fragile and unreliable!")

=== Raw LLM Output ===
**Question:**  
Which layer of the OSI model is primarily responsible for ensuring reliable, error-free delivery of data between end systems, including functions like segmentation, flow control, and error recovery?

**Options:**  
A) Network Layer  
B) Data Link Layer  
C) Transport Layer  
D) Session Layer  

**Correct Answer:**  
C) Transport Layer

THE PROBLEM: How do we extract the question, options, and correct
answer from this? Every model formats it differently. Every run might
be different. Parsing free-form text is fragile and unreliable!


### The XML Schema

We'll ask the model to output quiz questions in this **exact** format:

```xml
<quiz>
  <question>What does HTTP stand for?</question>
  <options>
    <option id="A">HyperText Transfer Protocol</option>
    <option id="B">High-Level Text Processing</option>
    <option id="C">Hyperlink and Text Transfer Process</option>
    <option id="D">Home Tool Transfer Protocol</option>
  </options>
  <correct>A</correct>
  <explanation>HTTP stands for HyperText Transfer Protocol, the foundation of web communication.</explanation>
</quiz>
```

Now Python can parse this reliably every time.

In [48]:
# The XML-structured system prompt — notice how PRECISE it is
QUIZ_XML_SYSTEM_PROMPT = """You are a Quiz Master that generates quiz questions in XML format.

You MUST respond with ONLY valid XML in this exact format — no other text before or after:

<quiz>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <explanation>Brief explanation of why the correct answer is correct.</explanation>
</quiz>

Rules:
- Always exactly 4 options labeled A, B, C, D
- The <correct> tag must contain exactly one letter: A, B, C, or D
- The <explanation> must explain WHY the answer is correct
- Questions should be challenging but fair for university students
- Wrong options should be plausible, not obviously silly
- Do NOT include any text outside the <quiz> tags
"""

# Generate a structured quiz question
response = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": "Generate a quiz question about operating systems."}
])

xml_text = response.choices[0].message.content
print("=== Structured XML Output ===")
print(xml_text)

=== Structured XML Output ===
<quiz>
  <question>Which CPU scheduling algorithm is theoretically optimal for minimizing the average waiting time of processes but is prone to the "convoy effect" where a long CPU-bound process can delay many short I/O-bound processes?</question>
  <options>
    <option id="A">Shortest Job Next (SJN) / Shortest Job First (SJF)</option>
    <option id="B">Round Robin (RR)</option>
    <option id="C">First-Come, First-Served (FCFS)</option>
    <option id="D">Priority Scheduling</option>
  </options>
  <correct>A</correct>
  <explanation>Shortest Job Next (SJN) is provably optimal for minimizing average waiting time in a batch system where job lengths are known. However, if a long CPU-bound job arrives before many short jobs, it causes the "convoy effect": all short jobs must wait behind the long one, increasing their waiting and response times. FCFS has no concept of job length, RR uses time slices to prevent this, and Priority Scheduling can suffer from s

In [49]:
def parse_quiz_xml(xml_string):
    """
    Parse a quiz question from XML format into a Python dict.

    Args:
        xml_string: XML string containing a quiz question

    Returns:
        Dict with keys: question, options (dict A-D), correct, explanation
        Returns None if parsing fails
    """
    try:
        # Clean up: strip any text before <quiz> and after </quiz>
        start = xml_string.find("<quiz>")
        end = xml_string.find("</quiz>")
        if start == -1 or end == -1:
            print("ERROR: Could not find <quiz>...</quiz> tags")
            return None

        clean_xml = xml_string[start:end + len("</quiz>")]
        root = ET.fromstring(clean_xml)

        question = root.find("question").text.strip()

        options = {}
        for opt in root.find("options").findall("option"):
            options[opt.get("id")] = opt.text.strip()

        correct = root.find("correct").text.strip()
        explanation = root.find("explanation").text.strip()

        # Validate
        if correct not in ["A", "B", "C", "D"]:
            print(f"WARNING: Invalid correct answer '{correct}', expected A-D")
            return None
        if len(options) != 4:
            print(f"WARNING: Expected 4 options, got {len(options)}")
            return None

        return {
            "question": question,
            "options": options,
            "correct": correct,
            "explanation": explanation
        }
    except ET.ParseError as e:
        print(f"XML Parse Error: {e}")
        print(f"Raw text was:\n{xml_string[:300]}...")
        return None
    except AttributeError as e:
        print(f"Missing XML element: {e}")
        return None

print("✓ parse_quiz_xml() function defined")

✓ parse_quiz_xml() function defined


In [50]:
# Parse the XML we generated and display it as structured data
quiz = parse_quiz_xml(xml_text)

if quiz:
    print("=== Parsed Quiz Question ===")
    print(f"\nQ: {quiz['question']}\n")
    for letter, text in quiz['options'].items():
        print(f"  {letter}) {text}")
    print(f"\nCorrect answer: {quiz['correct']}")
    print(f"Explanation: {quiz['explanation']}")
    print("\n→ Now we have clean, structured data we can use in code!")
else:
    print("Failed to parse! Let's see what went wrong...")
    print(xml_text)

=== Parsed Quiz Question ===

Q: Which CPU scheduling algorithm is theoretically optimal for minimizing the average waiting time of processes but is prone to the "convoy effect" where a long CPU-bound process can delay many short I/O-bound processes?

  A) Shortest Job Next (SJN) / Shortest Job First (SJF)
  B) Round Robin (RR)
  C) First-Come, First-Served (FCFS)
  D) Priority Scheduling

Correct answer: A
Explanation: Shortest Job Next (SJN) is provably optimal for minimizing average waiting time in a batch system where job lengths are known. However, if a long CPU-bound job arrives before many short jobs, it causes the "convoy effect": all short jobs must wait behind the long one, increasing their waiting and response times. FCFS has no concept of job length, RR uses time slices to prevent this, and Priority Scheduling can suffer from starvation but not specifically the convoy effect defined by job length.

→ Now we have clean, structured data we can use in code!


In [51]:
def play_quiz(quiz_data):
    """
    Play a single quiz question interactively.

    Args:
        quiz_data: Parsed quiz dict from parse_quiz_xml()

    Returns:
        True if answered correctly, False otherwise
    """
    print(f"\n{'=' * 60}")
    print(f"  {quiz_data['question']}")
    print('=' * 60)

    for letter, text in quiz_data['options'].items():
        print(f"  {letter}) {text}")

    answer = input("\nYour answer (A/B/C/D): ").strip().upper()

    if answer == quiz_data['correct']:
        print("\n✓ CORRECT!")
    else:
        print(f"\n✗ Incorrect. The correct answer is {quiz_data['correct']}.")

    print(f"\nExplanation: {quiz_data['explanation']}")
    return answer == quiz_data['correct']

print("✓ play_quiz() function defined")

✓ play_quiz() function defined


In [52]:
# What happens with a VAGUE prompt? Let's see it break.
print("=== Vague prompt — what could go wrong? ===")
response_vague = chat([
    {"role": "system", "content": "Generate quiz questions in XML."},  # TOO VAGUE!
    {"role": "user", "content": "Make a quiz."}
])

print(response_vague.choices[0].message.content[:500])
print("\n" + "=" * 70)
print("\nAttempting to parse...")
result = parse_quiz_xml(response_vague.choices[0].message.content)
if result is None:
    print("\n→ FAILED! A vague prompt leads to unpredictable format.")
    print("→ This is why the PRECISE system prompt with the EXACT schema matters.")
    print("→ Prompt engineering isn't about being fancy — it's about RELIABILITY.")
else:
    print("\n→ It worked this time, but try running it again — vague prompts are inconsistent.")

=== Vague prompt — what could go wrong? ===
```xml
<?xml version="1.0" encoding="UTF-8"?>
<quiz>
    <title>General Knowledge Quiz</title>
    <description>Test your knowledge with this mix of science, geography, and history questions.</description>
    <questions>
        <question id="q1" type="multiple_choice" difficulty="medium">
            <text>What is the powerhouse of the cell?</text>
            <options>
                <option id="a">Nucleus</option>
                <option id="b">Ribosome</option>
                <option id="


Attempting to parse...
Missing XML element: 'NoneType' object has no attribute 'text'

→ FAILED! A vague prompt leads to unpredictable format.
→ This is why the PRECISE system prompt with the EXACT schema matters.
→ Prompt engineering isn't about being fancy — it's about RELIABILITY.


In [53]:
# Full flow: generate → parse → play!
print("Let's play a quiz question!\n")

response = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": "Generate a quiz question about data structures."}
])

quiz = parse_quiz_xml(response.choices[0].message.content)
if quiz:
    play_quiz(quiz)

Let's play a quiz question!


  What is the average-case time complexity for searching an element in a balanced binary search tree?
  A) O(1)
  B) O(log n)
  C) O(n)
  D) O(n^2)

Your answer (A/B/C/D): C

✗ Incorrect. The correct answer is B.

Explanation: In a balanced binary search tree, the height is maintained to be logarithmic in the number of nodes (O(log n)), so a search operation traverses a path from the root to a leaf, taking time proportional to the height on average.


---

## Exercise 5: Thinking Models & Prompt Engineering

### Goal
Compare standard models with **thinking/reasoning models** that show their chain of thought. Then see how **prompt engineering** dramatically affects output quality.

### What Are Thinking Models?
Some models (like `openai/gpt-oss-120b`) have a built-in reasoning step:
1. The model first **THINKS** through the problem (chain-of-thought)
2. Then it produces the final answer
3. You can see the **reasoning tokens** — the model's "inner monologue"

This produces better answers for complex tasks, but uses **significantly more tokens** (and costs more).

> **Note:** If `gpt-oss-120b` is slow or unavailable, you can skip the thinking model cell — the prompt engineering exercises below work with any model.

In [54]:
# A task that benefits from reasoning
task = """Generate a quiz question about the difference between TCP and UDP
that tests DEEP understanding, not just definitions.
Present a realistic scenario where choosing the right protocol matters."""

# Standard model first
print("=== Standard Model (Step Flash) ===")
start = time.time()
response_standard = chat(
    [{"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
     {"role": "user", "content": task}],
    model=MODEL_FAST
)
time_standard = time.time() - start

quiz_standard = parse_quiz_xml(response_standard.choices[0].message.content)
if quiz_standard:
    print(f"Q: {quiz_standard['question']}")
    for k, v in quiz_standard['options'].items():
        print(f"  {k}) {v}")
print(f"\nTokens: {response_standard.usage.total_tokens} | Time: {time_standard:.1f}s")

=== Standard Model (Step Flash) ===
Q: You are designing the network layer for a real-time, interactive virtual reality (VR) training simulation where a user's head and hand movements must be transmitted to a remote server 50 times per second to update the virtual environment. The simulation cannot tolerate significant latency (delays), as even 100ms of lag causes severe motion sickness. However, occasional missed or out-of-order movement packets are acceptable, as the next update (20ms later) will provide a more recent and relevant state. Which transport protocol should you choose for this movement data stream, and why is it superior to the alternative in this specific context?
  A) TCP, because its guaranteed delivery and in-order sequencing ensure the virtual environment always has the complete and correct sequence of user movements, preventing state corruption.
  B) UDP, because its connectionless, low-overhead nature minimizes transmission latency and jitter. Retransmitting a stal

In [55]:
# Now the THINKING model — watch the reasoning tokens!
print("=== Thinking Model (GPT-OSS-120B) ===")
print("Generating... (this may take longer)\n")

start = time.time()
response_thinking = chat(
    [{"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
     {"role": "user", "content": task}],
    model=MODEL_THINKING,
    extra_body={
        "reasoning": {
            "effort": "high"
        }
    }
)
time_thinking = time.time() - start

# Access the reasoning (chain-of-thought) — the model's inner monologue
reasoning = getattr(response_thinking.choices[0].message, 'reasoning', None)

if reasoning:
    print("--- REASONING (model's inner thoughts) ---")
    # Show first 1500 chars of reasoning
    print(reasoning[:1500])
    if len(reasoning) > 1500:
        print(f"\n... [{len(reasoning)} total characters of reasoning]")
    print("\n--- END REASONING ---\n")
else:
    print("(No reasoning field returned — model may not support it via this provider)\n")

print("--- FINAL ANSWER ---")
quiz_thinking = parse_quiz_xml(response_thinking.choices[0].message.content)
if quiz_thinking:
    print(f"Q: {quiz_thinking['question']}")
    for k, v in quiz_thinking['options'].items():
        print(f"  {k}) {v}")
print(f"\nTokens: {response_thinking.usage.total_tokens} | Time: {time_thinking:.1f}s")

=== Thinking Model (GPT-OSS-120B) ===
Generating... (this may take longer)

--- REASONING (model's inner thoughts) ---
We need to output only XML with <quiz> root, containing <question>, <options> with 4 <option id="A"> etc, <correct>, <explanation>. Must be a deep understanding question, realistic scenario. Provide plausible options.

We must ensure no extra text outside <quiz>. Must be valid XML. Provide one question. Ensure correct answer is one letter.

Let's craft scenario: "A video streaming service must deliver live sports to millions with minimal latency, but occasional packet loss is acceptable. Which protocol is better?" Options: A: TCP, B: UDP, C: TCP with selective ACK, D: SCTP. Correct: B. Explanation: UDP is connectionless, no retransmission, lower latency, tolerates loss.

Make deep: ask about trade-offs: "Which of the following statements best describes why UDP is preferred in this scenario?" Provide options.

Ok produce.



--- END REASONING ---

--- FINAL ANSWER ---
Q

In [57]:
# Token usage comparison — side by side
print("=== Token Usage Comparison ===")
print(f"{'':30} {'Standard':>12} {'Thinking':>12}")
print("-" * 56)
print(f"{'Prompt tokens':30} {response_standard.usage.prompt_tokens:>12} {response_thinking.usage.prompt_tokens:>12}")
print(f"{'Completion tokens':30} {response_standard.usage.completion_tokens:>12} {response_thinking.usage.completion_tokens:>12}")
print(f"{'Total tokens':30} {response_standard.usage.total_tokens:>12} {response_thinking.usage.total_tokens:>12}")
print(f"{'Time (seconds)':30} {time_standard:>12.1f} {time_thinking:>12.1f}")

ratio = response_thinking.usage.total_tokens / max(response_standard.usage.total_tokens, 1)
print(f"\n→ Thinking model used {ratio:.1f}x more tokens!")
print("→ Reasoning is powerful but expensive — use it for tasks that need it.")
print("→ The thinking tokens are the model's 'chain of thought' — it reasons before answering.")

=== Token Usage Comparison ===
                                   Standard     Thinking
--------------------------------------------------------
Prompt tokens                           266          322
Completion tokens                      1186          406
Total tokens                           1452          728
Time (seconds)                         17.5         11.0

→ Thinking model used 0.5x more tokens!
→ Reasoning is powerful but expensive — use it for tasks that need it.
→ The thinking tokens are the model's 'chain of thought' — it reasons before answering.


### Prompt Engineering: Lazy vs. Engineered

The quality of your prompt **dramatically** affects the quality of the output. Let's prove it.

In [58]:
# ROUND 1: LAZY prompt
lazy_prompt = "Make a quiz question about sorting algorithms."

# ROUND 2: ENGINEERED prompt
engineered_prompt = """Generate a quiz question about sorting algorithms that:
- Tests understanding of TIME COMPLEXITY trade-offs (not just "what is quicksort")
- Presents a realistic scenario (e.g., choosing the right algorithm for a specific dataset)
- Has plausible distractors that represent common misconceptions
- The wrong answers should be algorithms that COULD seem correct but aren't optimal for the scenario
- Difficulty: university-level data structures course"""

print("=== LAZY PROMPT ===")
print(f"Prompt: \"{lazy_prompt}\"\n")
resp_lazy = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": lazy_prompt}
])
quiz_lazy = parse_quiz_xml(resp_lazy.choices[0].message.content)
if quiz_lazy:
    print(f"Q: {quiz_lazy['question']}")
    for k, v in quiz_lazy['options'].items():
        print(f"  {k}) {v}")

print(f"\n{'=' * 70}")
print("\n=== ENGINEERED PROMPT ===")
print(f"Prompt: \"{engineered_prompt[:80]}...\"\n")
resp_eng = chat([
    {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
    {"role": "user", "content": engineered_prompt}
])
quiz_eng = parse_quiz_xml(resp_eng.choices[0].message.content)
if quiz_eng:
    print(f"Q: {quiz_eng['question']}")
    for k, v in quiz_eng['options'].items():
        print(f"  {k}) {v}")

print("\n→ Compare: which question tests deeper understanding?")
print("→ Which has better distractors? More realistic scenario?")
print("→ The engineered prompt didn't cost more tokens — just more thought.")

=== LAZY PROMPT ===
Prompt: "Make a quiz question about sorting algorithms."

Q: Which of the following sorting algorithms is not based on comparisons between elements?
  A) QuickSort
  B) MergeSort
  C) HeapSort
  D) RadixSort


=== ENGINEERED PROMPT ===
Prompt: "Generate a quiz question about sorting algorithms that:
- Tests understanding of..."

Q: You are sorting a dataset of 1,000,000 integers where each element is at most 5 positions away from its correct sorted position (i.e., the array is "almost sorted"). Memory is constrained, so you must use an in-place algorithm (O(1) extra space). Which algorithm provides the best time complexity for this specific scenario?
  A) Quicksort (with a median-of-three pivot)
  B) Merge sort (using a standard top-down implementation)
  C) Heapsort
  D) Insertion sort

→ Compare: which question tests deeper understanding?
→ Which has better distractors? More realistic scenario?
→ The engineered prompt didn't cost more tokens — just more thought.


In [59]:
# Same engineered prompt across multiple models — compare quality
print("=== Multi-Model Comparison (same engineered prompt) ===")

for model_name in [MODEL_FAST, MODEL_MISTRAL, MODEL_NEMOTRON, MODEL_MINISTRAL]:
    resp = chat([
        {"role": "system", "content": QUIZ_XML_SYSTEM_PROMPT},
        {"role": "user", "content": engineered_prompt}
    ], model=model_name)

    quiz = parse_quiz_xml(resp.choices[0].message.content)
    print(f"\n{'=' * 70}")
    print(f"Model: {model_name}")
    print('=' * 70)
    if quiz:
        print(f"Q: {quiz['question']}")
        for k, v in quiz['options'].items():
            print(f"  {k}) {v}")
        print(f"Answer: {quiz['correct']}")
    else:
        print("Failed to parse XML output")
    print(f"Tokens: {resp.usage.total_tokens}")

=== Multi-Model Comparison (same engineered prompt) ===

Model: stepfun/step-3.5-flash:free
Q: You are tasked with sorting a dataset of 1 million integers that are already approximately in order—each element is at most 10 positions away from its correct sorted location. The system has limited memory, so you must sort in-place without allocating additional O(n) storage. Which algorithm provides the best asymptotic time complexity for this specific scenario?
  A) Quicksort with a median-of-three pivot selection
  B) Standard mergesort
  C) Heapsort
  D) Insertion sort
Answer: D
Tokens: 4089

Model: mistralai/mistral-small-2603
Q: A dataset of 5,000,000 integers is stored in an array where nearly all elements are identical (e.g., 4,999,990 values are '7' and only 10 are unique). The data must be sorted in-place with minimal additional memory. Which sorting algorithm is most appropriate for this scenario?
  A) Quicksort
  B) Mergesort
  C) Counting Sort
  D) Insertion Sort
Answer: D
Tokens

---

## Exercise 6: Study Buddy — Bring Your Own Knowledge (Capstone)

### Goal
Build a complete quiz game from **YOUR OWN study notes**! This ties together everything:

| Concept | From Exercise | Used Here |
|---------|--------------|----------|
| API calls & tokens | Exercise 1 | Every generation call |
| Conversation history | Exercise 2 | No repeated questions |
| System prompts | Exercise 3 | Quiz master + knowledge base |
| XML parsing | Exercise 4 | Parse every question |
| Prompt engineering | Exercise 5 | Quality of generated questions |

### How It Works
1. You upload (or paste) your study notes in markdown
2. The **entire text** is injected into the system prompt as context
3. The model generates quiz questions **from your material only**
4. Conversation history prevents repeated questions
5. You play the quiz interactively!

In [65]:
# OPTION 1: Upload a markdown/text file with your study notes
# (skip this cell and use the next one if upload doesn't work)

study_notes = None

try:
    from google.colab import files
    print("Upload your study notes (.md or .txt file):")
    print("(Or skip this cell and use the next one to paste text directly)\n")
    uploaded = files.upload()

    if uploaded:
        filename = list(uploaded.keys())[0]
        study_notes = uploaded[filename].decode('utf-8')
        print(f"\n✓ Loaded '{filename}' ({len(study_notes)} characters, ~{len(study_notes)//4} tokens)")
except Exception as e:
    print(f"Upload not available: {e}")
    print("Use the next cell to paste your notes directly.")

Upload your study notes (.md or .txt file):
(Or skip this cell and use the next one to paste text directly)



In [66]:
# OPTION 2: Paste your notes here (or use this sample if you don't have notes handy)

if study_notes is None:
    study_notes = """
# Friends TV Show

## Main Characters
Friends follows six close friends living in New York City.
- Rachel Green works in fashion and begins the series after leaving Barry at the altar
- Monica Geller is Ross’s sister and is known for being highly organized and competitive
- Ross Geller is a paleontologist and Monica’s older brother
- Chandler Bing is known for sarcasm and later marries Monica
- Joey Tribbiani is an actor famous for saying “How you doin’?”
- Phoebe Buffay is eccentric, plays guitar, and sings songs like “Smelly Cat”

## Relationships
Several major storylines focus on relationships.
- Ross and Rachel have an on-again, off-again romance throughout the series
- Monica and Chandler begin dating after London and later get married
- Joey briefly develops feelings for Rachel
- Phoebe marries Mike Hannigan

## Famous Places
A lot of the show takes place in familiar locations.
- Central Perk is the coffee shop where the group often hangs out
- Monica’s apartment is one of the main settings
- Joey and Chandler’s apartment is across the hall from Monica’s
- The show is set in Manhattan, New York City

## Fun Facts
- The series originally aired from 1994 to 2004
- The show has 10 seasons
- The six main characters are Rachel, Monica, Phoebe, Joey, Chandler, and Ross
- Chandler and Monica adopt twins at the end of the series
- Rachel famously “gets off the plane” in the finale

## Iconic Details
- Joey says “How you doin’?”
- Phoebe performs “Smelly Cat”
- Ross says “We were on a break!”
- The friends often sit on the orange couch in Central Perk
"""
    print("Using sample notes: 'TV Show Friends'")

print(f"\nNotes: {len(study_notes)} characters (~{len(study_notes)//4} tokens)")
print(f"\n=== Preview ===")
print(study_notes[:500])
if len(study_notes) > 500:
    print(f"\n... [{len(study_notes)} total characters]")

Using sample notes: 'TV Show Friends'

Notes: 1576 characters (~394 tokens)

=== Preview ===

# Friends TV Show

## Main Characters
Friends follows six close friends living in New York City.
- Rachel Green works in fashion and begins the series after leaving Barry at the altar
- Monica Geller is Ross’s sister and is known for being highly organized and competitive
- Ross Geller is a paleontologist and Monica’s older brother
- Chandler Bing is known for sarcasm and later marries Monica
- Joey Tribbiani is an actor famous for saying “How you doin’?”
- Phoebe Buffay is eccentric, plays gui

... [1576 total characters]


In [68]:
# Build the Study Buddy system prompt — note how we inject the FULL notes
STUDY_BUDDY_PROMPT = f"""You are a Study Buddy Quiz Master. Your job is to generate quiz questions
based ONLY on the study material provided below.

STUDY MATERIAL:
---
{study_notes}
---

RULES:
1. Generate questions ONLY from the material above — do NOT use outside knowledge
2. Test understanding and application, not just memorization
3. Make wrong options plausible (common misconceptions about these topics)
4. Vary the difficulty across questions
5. Do NOT repeat a question that has already been asked in this conversation
6. Cover different sections of the material across questions

Respond with ONLY valid XML in this exact format:

<quiz>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <explanation>Brief explanation referencing the study material.</explanation>
</quiz>
"""

# How big is this prompt?
prompt_tokens_est = len(STUDY_BUDDY_PROMPT) // 4
notes_tokens_est = len(study_notes) // 4
print(f"System prompt size: ~{prompt_tokens_est} tokens")
print(f"  of which study notes: ~{notes_tokens_est} tokens")
print(f"\n→ This system prompt is sent with EVERY request.")
print(f"→ Larger notes = more tokens per call = higher cost.")
print(f"→ This is exactly the problem that RAG solves (next session)!")

System prompt size: ~632 tokens
  of which study notes: ~394 tokens

→ This system prompt is sent with EVERY request.
→ Larger notes = more tokens per call = higher cost.
→ This is exactly the problem that RAG solves (next session)!


In [71]:
# Generate and play your first Study Buddy question!
quiz_session = Conversation(model=MODEL_FAST, system_prompt=STUDY_BUDDY_PROMPT)

print("Generating your first quiz question from your notes...\n")
xml_response, usage = quiz_session.say("Generate a quiz question.")

quiz = parse_quiz_xml(xml_response)
if quiz:
    play_quiz(quiz)
    #print(f"\n[Tokens used: {usage.total_tokens}]")
    print(f"\n[Tokens used: {usage}]") # Corrected
else:
    print("Failed to parse. Raw response:")
    print(xml_response)

Generating your first quiz question from your notes...


  Which couple in Friends begins their romantic relationship after a trip to London?
  A) Ross and Rachel
  B) Monica and Chandler
  C) Joey and Rachel
  D) Phoebe and Mike

Your answer (A/B/C/D): b

✓ CORRECT!

Explanation: The study material specifies in the Relationships section: "Monica and Chandler begin dating after London and later get married."

[Tokens used: 2608]


In [75]:
# Play 3 more rounds — the model won't repeat questions because
# the full conversation history is sent each time (Exercise 2!)
score = 0
rounds = 3

for round_num in range(1, rounds + 1):
    print(f"\n{'#' * 60}")
    print(f"   ROUND {round_num} of {rounds}")
    print('#' * 60)

    xml_response, usage = quiz_session.say(
        "Generate another quiz question on a DIFFERENT topic from the previous ones."
    )
    quiz = parse_quiz_xml(xml_response)

    if quiz:
        correct = play_quiz(quiz)
        if correct:
            score += 1
        #print(f"\n[Tokens this round: {usage.total_tokens}]")
        print(f"\n[Tokens this round: {usage}]") #Corrected
    else:
        print("Failed to generate a valid question. Skipping...")

print(f"\n{'=' * 60}")
print(f"   FINAL SCORE: {score}/{rounds}")
print(f"   Messages in conversation: {len(quiz_session.get_history())}")
print('=' * 60)


############################################################
   ROUND 1 of 3
############################################################

  What significant action does Rachel Green take in the series finale of Friends?
  A) She gets on the plane to Paris for a job.
  B) She gets off the plane to stay with Ross.
  C) She decides to move to London with Monica and Chandler.
  D) She reunites with Barry and marries him.

Your answer (A/B/C/D): b

✓ CORRECT!

Explanation: The study material specifies in the Fun Facts section: "Rachel famously 'gets off the plane' in the finale."

[Tokens this round: 12094]

############################################################
   ROUND 2 of 3
############################################################

  Which couple adopts twins at the end of the Friends series?
  A) Ross and Rachel
  B) Monica and Chandler
  C) Joey and Phoebe
  D) Phoebe and Mike

Your answer (A/B/C/D): b

✓ CORRECT!

Explanation: The study material states in the Fun Facts sec

### Token Scaling

Notice how each round uses **more tokens** than the last. That's because the **full conversation history** grows with each exchange.

This is the fundamental trade-off:
- **More history** = better context, no repeated questions
- **More history** = more tokens per call, higher cost, eventually hits context window limit

In production, you'd address this with:
- **Sliding window** — only keep the last N messages
- **Summarization** — summarize old turns into a compact form
- **RAG** — retrieve only relevant context instead of sending everything (next session!)

In [76]:
# Visualize how tokens scale with conversation length
print("=== Token Scaling Across the Conversation ===")
print("(estimated from character count)\n")

history = quiz_session.get_history()
running_chars = 0
for i, msg in enumerate(history):
    running_chars += len(msg["content"])
    estimated_tokens = running_chars // 4
    bar = "█" * (estimated_tokens // 100)
    role = msg["role"]
    print(f"  Msg {i+1:2d} ({role:10s}): ~{estimated_tokens:5d} tokens  {bar}")

print(f"\n→ The system prompt ({len(STUDY_BUDDY_PROMPT)//4}+ tokens) is re-sent every single time.")
print("→ By round 3, we're sending thousands of tokens just in conversation history.")

=== Token Scaling Across the Conversation ===
(estimated from character count)

  Msg  1 (system    ): ~  632 tokens  ██████
  Msg  2 (user      ): ~  639 tokens  ██████
  Msg  3 (assistant ): ~  765 tokens  ███████
  Msg  4 (user      ): ~  784 tokens  ███████
  Msg  5 (assistant ): ~  884 tokens  ████████
  Msg  6 (user      ): ~  903 tokens  █████████
  Msg  7 (assistant ): ~ 1007 tokens  ██████████
  Msg  8 (user      ): ~ 1026 tokens  ██████████
  Msg  9 (user      ): ~ 1045 tokens  ██████████
  Msg 10 (assistant ): ~ 1165 tokens  ███████████
  Msg 11 (user      ): ~ 1184 tokens  ███████████
  Msg 12 (user      ): ~ 1202 tokens  ████████████
  Msg 13 (assistant ): ~ 1351 tokens  █████████████
  Msg 14 (user      ): ~ 1370 tokens  █████████████
  Msg 15 (assistant ): ~ 1487 tokens  ██████████████
  Msg 16 (user      ): ~ 1505 tokens  ███████████████
  Msg 17 (assistant ): ~ 1649 tokens  ████████████████

→ The system prompt (632+ tokens) is re-sent every single time.
→ By round 3, 

In [78]:
# Try with a different model — same notes, different quiz master
print(f"=== Same notes, different model: {MODEL_MISTRAL} ===")

quiz_session_ds = Conversation(model=MODEL_MISTRAL, system_prompt=STUDY_BUDDY_PROMPT)

xml_response, usage = quiz_session_ds.say("Generate a challenging quiz question.")
quiz = parse_quiz_xml(xml_response)
if quiz:
    play_quiz(quiz)
    #print(f"\n[Tokens: {usage.total_tokens}]")
    print(f"\n[Tokens: {usage}]") #Corrected

=== Same notes, different model: mistralai/mistral-small-2603 ===

  Which of the following relationships is NOT explicitly mentioned in the study material as a romantic pairing that develops between the main characters?
  A) Ross and Rachel
  B) Monica and Chandler
  C) Phoebe and Mike
  D) Joey and Monica

Your answer (A/B/C/D): d

✓ CORRECT!

Explanation: According to the study material, the explicitly mentioned romantic pairings are Ross and Rachel, Monica and Chandler, and Phoebe and Mike. Joey and Monica are not mentioned as a romantic pairing.

[Tokens: 771]


---

## Key Takeaways

### What We Learned

1. **API Basics (Exercise 1)** — LLMs are accessed via the Chat Completions API. Every call has: messages in, response out, token usage. Different models have different strengths, speeds, and costs.

2. **Statelessness (Exercise 2)** — LLMs have NO memory between calls. YOU manage conversation history by sending all messages each time. More history = more tokens per call.

3. **System Prompts (Exercise 3)** — The system prompt is your primary control mechanism. Same question + different system prompt = dramatically different output.

4. **Structured Output (Exercise 4)** — XML schemas give you reliable, parseable output. A precise schema in the prompt is essential. Always validate and handle parsing failures.

5. **Thinking Models (Exercise 5)** — Reasoning models show chain-of-thought and produce better answers for complex tasks, but burn significantly more tokens. Prompt engineering matters as much as model selection.

6. **Knowledge in Context (Exercise 6)** — You can inject documents into the prompt as context. Token costs scale with document size + conversation length. This works but doesn't scale.

### What's Next

In the next session, we'll tackle the scaling problem from Exercise 6 with **Retrieval-Augmented Generation (RAG)**:
- Instead of sending ALL your notes in the prompt, we'll use **embeddings** to convert text into vectors
- A **vector database** will store and search these embeddings
- Only the **relevant sections** get retrieved and sent to the model
- This is how production AI systems work at scale — ChatGPT, Perplexity, and enterprise AI assistants all use RAG

## Bonus Challenges (Optional)

1. **Multi-topic quiz** — Upload notes from multiple subjects, ask the model to generate mixed quizzes
2. **Difficulty levels** — Modify the system prompt to accept a difficulty parameter (easy/medium/hard) and test whether the model actually follows it
3. **Model tournament** — Run the same 10 questions through all 3 models, compare which produces the best questions
4. **Prompt optimization** — Experiment with adding tags like `<difficulty>`, `<topic>`, `<hint>` to the XML schema
5. **Token budget** — Build a version that summarizes old conversation turns to stay under a token budget

In [ ]:
# Your experiments here!

**Multi-topic quiz** — Upload notes from multiple subjects, ask the model to generate mixed quizzes

In [89]:
study_notes = """
# Friends

## Main Characters
- Rachel Green works in fashion and leaves Barry at the altar at the start of the series
- Monica Geller is Ross's sister and is very organized and competitive
- Ross Geller is a paleontologist
- Chandler Bing is sarcastic and later marries Monica
- Joey Tribbiani is an actor known for saying "How you doin'?"
- Phoebe Buffay sings "Smelly Cat"

## Relationships
- Ross and Rachel have an on-again, off-again relationship
- Monica and Chandler begin dating after London and later get married
- Phoebe marries Mike Hannigan

## Places and Facts
- Central Perk is the group's favorite coffee shop
- The show is set in New York City
- Friends originally aired from 1994 to 2004
- The show has 10 seasons


# Stranger Things

## Main Characters
- Eleven has psychokinetic powers
- Mike Wheeler is one of the central kids and is close to Eleven
- Dustin Henderson is known for his humor and intelligence
- Lucas Sinclair is practical and cautious
- Will Byers disappears in season 1
- Jim Hopper is Hawkins' police chief

## Key Story Elements
- The story is set in Hawkins, Indiana
- The Upside Down is a dark parallel dimension
- The Demogorgon is one of the first major monsters
- Vecna becomes a major villain later in the series

## Facts
- Stranger Things premiered in 2016
- The show mixes science fiction, horror, and coming-of-age themes


# The Big Bang Theory

## Main Characters
- Sheldon Cooper is highly intelligent and socially awkward
- Leonard Hofstadter is a physicist and Sheldon's roommate at the start
- Penny lives across the hall and becomes part of the group
- Howard Wolowitz is an engineer
- Raj Koothrappali is an astrophysicist
- Amy Farrah Fowler later becomes Sheldon's partner
- Bernadette Rostenkowski later marries Howard

## Key Details
- Much of the show takes place in Pasadena, California
- The group often spends time in Apartment 4A
- The Cheesecake Factory is one of Penny's workplaces

## Facts
- The Big Bang Theory premiered in 2007
- The show has 12 seasons
"""

In [90]:
STUDY_BUDDY_PROMPT = f"""You are a Study Buddy Quiz Master.
Your job is to generate quiz questions based ONLY on the study material provided below.

STUDY MATERIAL:
---
{study_notes}
---

RULES:
1. Generate questions ONLY from the material above — do NOT use outside knowledge
2. Create a multi-topic quiz using all three topics: Friends, Stranger Things, and The Big Bang Theory
3. Mix the topics across questions instead of focusing on only one topic
4. Test understanding, not just memorization
5. Make wrong options plausible
6. Vary the difficulty across questions
7. Do NOT repeat a question that has already been asked in this conversation
8. Cover different sections of the material across questions

Respond with ONLY valid XML in this exact format:

<quiz>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <explanation>Brief explanation referencing the study material.</explanation>
</quiz>
"""

In [95]:
quiz_session = Conversation(model=MODEL_FAST, system_prompt=STUDY_BUDDY_PROMPT)

xml_response, usage = quiz_session.say(
    "Generate one multiple-choice quiz question from one of the three topics. Make sure topic coverage stays balanced across the quiz."
)

quiz = parse_quiz_xml(xml_response)
if quiz:
    play_quiz(quiz)


  Which of the following relationships in Friends develops after the group travels to London?
  A) Ross and Rachel
  B) Monica and Chandler
  C) Phoebe and Mike
  D) Rachel and Joey

Your answer (A/B/C/D): b

✓ CORRECT!

Explanation: Monica and Chandler begin dating after London and later get married.


**Difficulty levels** — Modify the system prompt to accept a difficulty parameter (easy/medium/hard) and test whether the model actually follows it

In [96]:
def build_study_buddy_prompt(study_notes, difficulty="medium"):
    return f"""You are a Study Buddy Quiz Master.
Your job is to generate quiz questions based ONLY on the study material provided below.

STUDY MATERIAL:
---
{study_notes}
---

DIFFICULTY LEVEL: {difficulty}

DIFFICULTY RULES:
- easy: ask direct, obvious questions; focus on basic facts, names, places, simple relationships
- medium: ask questions that require connecting two facts or understanding context
- hard: ask more challenging questions that require comparison, inference, or distinguishing between similar facts

GENERAL RULES:
1. Generate questions ONLY from the material above — do NOT use outside knowledge
2. Follow the requested difficulty level strictly
3. Always provide exactly 4 options: A, B, C, D
4. Make wrong options plausible
5. Do NOT repeat a question already asked in this conversation
6. Cover different sections of the material across questions
7. Respond with ONLY valid XML in this exact format:

<quiz>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <explanation>Brief explanation referencing the study material.</explanation>
</quiz>
"""

In [97]:
difficulty = "easy"   #"medium" or "hard"

study_prompt = build_study_buddy_prompt(study_notes, difficulty=difficulty)
quiz_session = Conversation(model=MODEL_FAST, system_prompt=study_prompt)

xml_response, usage = quiz_session.say(
    f"Generate one {difficulty} quiz question."
)

quiz = parse_quiz_xml(xml_response)
if quiz:
    play_quiz(quiz)
else:
    print(xml_response)


  Which character from Friends is known for saying "How you doin'?"
  A) Chandler Bing
  B) Ross Geller
  C) Joey Tribbiani
  D) Phoebe Buffay

Your answer (A/B/C/D): c

✓ CORRECT!

Explanation: The study material states under "Main Characters" for Friends: "Joey Tribbiani is an actor known for saying 'How you doin'?'"


In [102]:
for difficulty in ["easy", "medium", "hard"]:
    print(f"\n{'=' * 70}")
    print(f"DIFFICULTY: {difficulty.upper()}")
    print('=' * 70)

    prompt = build_study_buddy_prompt(study_notes, difficulty=difficulty)
    session = Conversation(model=MODEL_FAST, system_prompt=prompt)

    xml_response, usage = session.say(
        f"Generate one {difficulty} quiz question from the notes."
    )

    quiz = parse_quiz_xml(xml_response)
    if quiz:
        print("Q:", quiz["question"])
        for k, v in quiz["options"].items():
            print(f"  {k}) {v}")
        print("Correct:", quiz["correct"])
        print("Explanation:", quiz["explanation"])
        print("Tokens:", usage)
    else:
        print("FAILED TO PARSE")
        print(xml_response)


DIFFICULTY: EASY
Q: Which character in Friends is known for saying "How you doin'?"?
  A) Joey Tribbiani
  B) Chandler Bing
  C) Ross Geller
  D) Mike Wheeler
Correct: A
Explanation: The study material states: "Joey Tribbiani is an actor known for saying 'How you doin'?'"
Tokens: 2541

DIFFICULTY: MEDIUM
Q: Based on the provided information, in which location did Monica and Chandler's romantic relationship begin?
  A) New York City
  B) London
  C) Pasadena, California
  D) Hawkins, Indiana
Correct: B
Explanation: The study material states for Friends: "Monica and Chandler begin dating after London." This connects the specific event of their relationship starting to the location of London.
Tokens: 1388

DIFFICULTY: HARD
Q: Which television series includes both a character who is a police chief and a concept of a parallel dimension?
  A) Friends
  B) Stranger Things
  C) The Big Bang Theory
  D) None of the above
Correct: B
Explanation: Stranger Things features Jim Hopper as Hawkins' p

**Model tournament** — Run the same 3 questions through all 4 models, compare which produces the best questions

In [109]:
questions = [
    "Generate a quiz question about The Big Bang Theory.",
    "Generate a mixed-topic quiz question using any of the three shows.",
    "Generate a hard mixed-topic quiz question that requires comparing two shows."
]

models = [MODEL_FAST, MODEL_MISTRAL, MODEL_NEMOTRON, MODEL_MINISTRAL]

for model_name in models:
    print(f"\n{'=' * 80}")
    print(f"MODEL: {model_name}")
    print('=' * 80)

    for i, question in enumerate(questions, start=1):
        print(f"\n--- QUESTION {i} ---")
        response = chat(
            [
                {"role": "system", "content": STUDY_BUDDY_PROMPT},
                {"role": "user", "content": question}
            ],
            model=model_name
        )

        xml_text = response.choices[0].message.content
        quiz = parse_quiz_xml(xml_text)

        if quiz:
            print("Q:", quiz["question"])
            for k, v in quiz["options"].items():
                print(f"  {k}) {v}")
            print("Correct:", quiz["correct"])
            print("Tokens:", response.usage.total_tokens)
        else:
            print("Failed to parse XML")
            print(xml_text[:500])


MODEL: stepfun/step-3.5-flash:free

--- QUESTION 1 ---
Q: In The Big Bang Theory, who becomes Sheldon Cooper's partner later in the series?
  A) Penny
  B) Bernadette Rostenkowski
  C) Amy Farrah Fowler
  D) Raj Koothrappali
Correct: C
Tokens: 2266

--- QUESTION 2 ---
Q: Which of the following locations is associated with the show Friends?
  A) Central Perk
  B) The Upside Down
  C) Apartment 4A
  D) The Cheesecake Factory
Correct: A
Tokens: 5520

--- QUESTION 3 ---
Q: Which of the following statements correctly compares the settings and premiere years of Friends and Stranger Things?
  A) Friends is set in New York City and premiered in 1994; Stranger Things is set in Hawkins, Indiana and premiered in 2016.
  B) Friends is set in New York City and premiered in 2016; Stranger Things is set in Hawkins, Indiana and premiered in 1994.
  C) Friends is set in Hawkins, Indiana and premiered in 1994; Stranger Things is set in New York City and premiered in 2016.
  D) Both are set in New York 

**Prompt optimization** — Experiment with adding tags like `<difficulty>`, `<topic>`, `<hint>` to the XML schema

In [115]:
QUIZXMLSYSTEMPROMPT_V2 = """
You are a Quiz Master that generates quiz questions in XML format.

You MUST respond with ONLY valid XML in this exact format:
<quiz>
  <topic>Topic name</topic>
  <difficulty>easy</difficulty>
  <question>Your question here</question>
  <options>
    <option id="A">First option</option>
    <option id="B">Second option</option>
    <option id="C">Third option</option>
    <option id="D">Fourth option</option>
  </options>
  <correct>A</correct>
  <hint>A short hint that helps without revealing the answer.</hint>
  <explanation>Brief explanation of why the correct answer is correct.</explanation>
</quiz>

Rules:
- Always output exactly 4 options labeled A, B, C, D
- The <correct> tag must contain exactly one letter: A, B, C, or D
- The <difficulty> tag must be one of: easy, medium, hard
- The <topic> tag must be short and specific
- The <hint> tag must help the student think, but must not reveal the answer directly
- The <explanation> tag must explain WHY the answer is correct
- Questions should be challenging but fair for university students
- Wrong options should be plausible
- Do NOT include any text before or after the XML
"""

In [116]:
def parse_quiz_xml_v2(xml_string):
    try:
        start = xml_string.find("<quiz>")
        end = xml_string.find("</quiz>")

        if start == -1 or end == -1:
            print("ERROR: Could not find <quiz>...</quiz> tags")
            return None

        clean_xml = xml_string[start:end + len("</quiz>")]
        root = ET.fromstring(clean_xml)

        topic = root.find("topic").text.strip()
        difficulty = root.find("difficulty").text.strip().lower()
        question = root.find("question").text.strip()
        hint = root.find("hint").text.strip()
        correct = root.find("correct").text.strip()
        explanation = root.find("explanation").text.strip()

        options = {}
        for opt in root.find("options").findall("option"):
            options[opt.get("id")] = opt.text.strip()

        if difficulty not in {"easy", "medium", "hard"}:
            print("WARNING: difficulty must be easy, medium, or hard")
            return None

        if correct not in {"A", "B", "C", "D"}:
            print("WARNING: correct answer must be A, B, C, or D")
            return None

        if len(options) != 4:
            print(f"WARNING: Expected 4 options, got {len(options)}")
            return None

        return {
            "topic": topic,
            "difficulty": difficulty,
            "question": question,
            "options": options,
            "correct": correct,
            "hint": hint,
            "explanation": explanation,
        }

    except ET.ParseError as e:
        print("XML Parse Error:", e)
        return None
    except AttributeError as e:
        print("Missing XML element:", e)
        return None

In [121]:
response = chat(
    [
        {"role": "system", "content": QUIZXMLSYSTEMPROMPT_V2},
        {"role": "user", "content": "Generate a hard quiz question about computer networks."}
    ],
    model=MODEL_FAST
)

xml_text = response.choices[0].message.content
quiz = parse_quiz_xml_v2(xml_text)

print(xml_text)
print("-" * 60)
print(quiz)

<quiz>
  <topic>BGP Path Selection</topic>
  <difficulty>hard</difficulty>
  <question>In BGP, when selecting the best path among multiple advertisements for the same prefix, which attribute has the highest precedence in the standard decision process as defined by RFC 4271?</question>
  <options>
    <option id="A">AS_PATH length</option>
    <option id="B">LOCAL_PREF</option>
    <option id="C">MED (Multi-Exit Discriminator)</option>
    <option id="D">ORIGIN</option>
  </options>
  <correct>B</correct>
  <hint>This attribute is set within an autonomous system and is not propagated to external BGP peers.</hint>
  <explanation>LOCAL_PREF is the first attribute evaluated in the BGP best path selection algorithm per RFC 4271. It indicates the degree of preference for a route within the local AS, with higher values preferred, and is used to influence outbound traffic without being advertised to external peers.</explanation>
</quiz>
---------------------------------------------------------

**Token budget** — Build a version that summarizes old conversation turns to stay under a token budget